### Cria dispositivo_hora e dispositivo_dia com MERGE pela chave de grão, e perfil_horario com CREATE OR REPLACE. Comportamento usa só leituras sem pico e sem sentinela; as contagens de problemas usam todas.

In [0]:
# 08 · Gold: agregações
# **Fase 3 · Gold** | Entrada: `silver.telemetria_clean`
# Tabelas: `gold.dispositivo_hora`, `gold.dispositivo_dia`, `gold.perfil_horario`.
# Regras:
# - Métricas de comportamento (médias, máx., mín.): só leituras sem sentinela e sem pico.
# - Contagens de problemas: todas as leituras.
# - `dispositivo_hora` e `dispositivo_dia`: `MERGE` pela chave de grão (idempotente).
# - `perfil_horario`: snapshot com `CREATE OR REPLACE`.

In [0]:
CATALOG = "project_lakehouse_telemetria_ele"
SILVER = f"{CATALOG}.silver.telemetria_clean"
GOLD = f"{CATALOG}.gold"
 
C_DEV, C_TS = "device_id", "event_ts"
C_TEMP, C_VOLT, C_CUR, C_ALARM = "temperatura_c", "tensao_v", "corrente_a", "alarme"   # confirmado no DESCRIBE
F_PICO, F_TENS, F_SENT = "temp_pico", "tensao_ausente", "corrente_sentinela"
 
# Mesma expressão definida no notebook 07
ALARME_ATIVO = f"""(CASE WHEN {C_ALARM} = 'SOBRETEMP' THEN 1 ELSE 0 END)"""   # valores reais: OK | SOBRETEMP
 
def fi(c):
    return f"COALESCE(CAST({c} AS INT),0)"
 
OK = f"({fi(F_PICO)}=0 AND {fi(F_SENT)}=0)"   # leitura utilizável para comportamento
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
 
METRICAS = f"""
  COUNT(*) AS leituras,
  SUM(CASE WHEN {OK} THEN 1 ELSE 0 END) AS leituras_validas,
  ROUND(AVG(CASE WHEN {OK} THEN {C_TEMP} END),2) AS temp_media,
  MAX(CASE WHEN {OK} THEN {C_TEMP} END) AS temp_max,
  MIN(CASE WHEN {OK} THEN {C_TEMP} END) AS temp_min,
  ROUND(AVG(CASE WHEN {OK} THEN {C_VOLT} END),2) AS tensao_media,
  ROUND(AVG(CASE WHEN {OK} THEN {C_CUR} END),2) AS corrente_media,
  SUM({fi(F_PICO)}) AS qtd_picos,
  SUM({ALARME_ATIVO}) AS qtd_alarmes,
  SUM({fi(F_TENS)}) AS qtd_tensao_ausente,
  SUM({fi(F_SENT)}) AS qtd_corrente_sentinela
"""
 
def upsert(view, table, keys):
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} AS SELECT * FROM {view} WHERE 1=0")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    spark.sql(f"""
        MERGE INTO {table} t USING {view} s ON {cond}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(table, "->", spark.table(table).count(), "linhas")

In [0]:
## 1. `gold.dispositivo_hora` (device_id × hora)
 
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_hora AS
SELECT {C_DEV} AS device_id, date_trunc('hour', {C_TS}) AS hora_ts, {METRICAS}
FROM {SILVER}
GROUP BY {C_DEV}, date_trunc('hour', {C_TS})
""")
upsert("v_hora", f"{GOLD}.dispositivo_hora", ["device_id", "hora_ts"])

In [0]:
## 2. `gold.dispositivo_dia` (device_id × dia)
 
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_dia AS
SELECT {C_DEV} AS device_id, data_evento, {METRICAS}
FROM {SILVER}
GROUP BY {C_DEV}, data_evento
""")
upsert("v_dia", f"{GOLD}.dispositivo_dia", ["device_id", "data_evento"])
 

In [0]:
## 3. `gold.perfil_horario` (hora do dia 0–23)
# Retoma o achado da Fase 1 (52,7 °C às 0h, 67,7 °C às 12h). Traz a média com e sem picos para comparar.
 
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.perfil_horario AS
SELECT HOUR({C_TS}) AS hora_do_dia,
       COUNT(*) AS leituras,
       ROUND(AVG({C_TEMP}),2) AS temp_media_todas,
       ROUND(AVG(CASE WHEN {OK} THEN {C_TEMP} END),2) AS temp_media_filtrada,
       SUM({fi(F_PICO)}) AS qtd_picos,
       SUM({ALARME_ATIVO}) AS qtd_alarmes
FROM {SILVER}
GROUP BY HOUR({C_TS})
""")
display(spark.table(f"{GOLD}.perfil_horario").orderBy("hora_do_dia"))

In [0]:
## 4. Conferência rápida (a validação completa está no notebook 10)
 
display(spark.sql(f"""
SELECT SUM(leituras) AS leituras, SUM(qtd_picos) AS picos, SUM(qtd_alarmes) AS alarmes,
       SUM(qtd_tensao_ausente) AS tensao_ausente, SUM(qtd_corrente_sentinela) AS corrente_sentinela,
       COUNT(*) AS linhas_tabela
FROM {GOLD}.dispositivo_hora
"""))